# Project Setup

- Team member: Chidinma Obi-Okoye
- Date: Wed 1 April, 2026
- Dataset version/source: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud?resource=download
- Objective for this notebook run:


In [6]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os


In [7]:
# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [8]:
# Paths
DATA_PATH = '../../data/'  # update as needed
ARTIFACT_PATH = '../artifacts/'


# 1) Data Understanding and Problem Framing

## Dataset
- **Name:** Credit Card Fraud Detection
- **Source:** Kaggle — ULB Machine Learning Group
- **File:** `creditcard.csv`
- **Size:** 284,807 transactions × 31 features
- **Features:** V1–V28 (PCA-transformed, anonymous), `Time`, `Amount`, `Class`
- **Target:** `Class` — 0 = legitimate, 1 = fraudulent
- **Fraud rate:** 492 / 284,807 ≈ 0.17%

## Anomaly Definition
A fraudulent transaction is defined as an anomaly. Normal behavior is the distribution 
of legitimate transactions. The goal is to learn the density of normal transactions and 
flag low-probability samples as anomalies at inference time.

## Framing: Semi-Supervised Anomaly Detection
Models are trained on normal samples only (no fraud labels during training).  
Fraud labels are used exclusively for threshold tuning and evaluation.

This approach is justified because:
1. Extreme class imbalance (99.83% normal) makes binary classification unreliable.
2. Fraud patterns are evolving; modeling normality generalizes better to novel anomalies.
3. Aligns with the course-defined semi-supervised anomaly detection procedure.

Prediction rule:
- f(x) = 1 (anomaly) if p(x) < ε
- f(x) = 0 (normal) otherwise

## Cost Assumptions
| Error Type | Business Impact |
|---|---|
| False Negative (missed fraud) | High — direct financial loss |
| False Positive (flagged legitimate) | Moderate — customer friction |

Primary metric: **Recall** (fraud class), reported alongside F1-score.  
Accuracy is not used as a primary metric due to class imbalance.

## Data Split Strategy
- Train: 60% of normal samples only
- Validation: 20% of normal + 50% of fraud samples
- Test: 20% of normal + 50% of fraud samples

In [9]:
# Load dataset
df = pd.read_csv(DATA_PATH + 'creditcard.csv')
df.head()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [11]:
# Verify class distribution (evidence for framing justification)
print("Shape:", df.shape)
print("\nClass distribution:")
print(df['Class'].value_counts())
print("\nFraud rate: {:.4f}%".format(df['Class'].mean() * 100))

Shape: (284807, 31)

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Fraud rate: 0.1727%


# 2) Data Quality Checks


In [12]:
# Missingness, duplicates, type checks, range checks
# AD-02: Dataset intake summary
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

Shape: (284807, 31)

Column types:
Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Missing values:
Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtyp

In [15]:
# Scale sumamry
print("Feature value ranges:")
df.describe().loc[['min', 'max', 'mean', 'std']]

Feature value ranges:


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
min,0.000000,-5.640751e+01,-7.271573e+01,-4.832559e+01,-5.683171e+00,-1.137433e+02,-2.616051e+01,-4.355724e+01,-7.321672e+01,-1.343407e+01,...,-3.483038e+01,-1.093314e+01,-4.480774e+01,-2.836627e+00,-1.029540e+01,-2.604551e+00,-2.256568e+01,-1.543008e+01,0.000000,0.000000
max,172792.000000,2.454930e+00,2.205773e+01,9.382558e+00,1.687534e+01,3.480167e+01,7.330163e+01,1.205895e+02,2.000721e+01,1.559499e+01,...,2.720284e+01,1.050309e+01,2.252841e+01,4.584549e+00,7.519589e+00,3.517346e+00,3.161220e+01,3.384781e+01,25691.160000,1.000000
mean,94813.859575,1.175161e-15,3.384974e-16,-1.379537e-15,2.094852e-15,1.021879e-15,1.494498e-15,-5.620335e-16,1.149614e-16,-2.414189e-15,...,1.628620e-16,-3.576577e-16,2.618565e-16,4.473914e-15,5.109395e-16,1.686100e-15,-3.661401e-16,-1.227452e-16,88.349619,0.001727
std,47488.145955,1.958696e+00,1.651309e+00,1.516255e+00,1.415869e+00,1.380247e+00,1.332271e+00,1.237094e+00,1.194353e+00,1.098632e+00,...,7.345240e-01,7.257016e-01,6.244603e-01,6.056471e-01,5.212781e-01,4.822270e-01,4.036325e-01,3.300833e-01,250.120109,0.041527


In [16]:
# Label and anomaly ratio
counts = df['Class'].value_counts()
ratio = df['Class'].mean() * 100
print(f"Normal transactions:    {counts[0]}")
print(f"Fraudulent transactions: {counts[1]}")
print(f"Anomaly ratio:           {ratio:.4f}%")

Normal transactions:    284315
Fraudulent transactions: 492
Anomaly ratio:           0.1727%


## Data Dictionary

| Feature | Type | Meaning | Scale | Missingness |
|---|---|---|---|---|
| Time | float64 | Seconds elapsed since first transaction | Raw (0 to ~172,000) | None |
| V1–V28 | float64 | PCA-transformed anonymized features | Standardized (~-30 to 30) | None |
| Amount | float64 | Transaction amount in euros | Raw (0 to ~25,000) | None |
| Class | int64 | 0 = legitimate, 1 = fraudulent | Binary | None |

## Label Availability
Labels are available for all 284,807 transactions.
Labels are used **only for evaluation**, not training (semi-supervised setup).

## Anomaly Ratio
- Normal: 284,315 (99.83%)
- Fraud: 492 (0.17%)
- Implication: accuracy is not a valid metric; recall and F1 are used instead.

## Scale Notes
- V1–V28 are already PCA-transformed and approximately standardized
- `Amount` and `Time` are raw and will require scaling before model input

# 3) EDA for Imbalance and Feature Behavior


In [ ]:
# Class distribution
# Feature distributions
# Correlation heatmap


# 4) Preprocessing Pipeline


In [ ]:
# Split train/val/test
# Fit transforms on train only
# Save transformed datasets


# 5) Statistical Anomaly Detection Method


In [ ]:
# Implement robust z-score or Gaussian density detector
# Produce anomaly_score and pred_label


# 6) Initial Threshold Candidates


In [ ]:
# Plot score distributions
# Propose 3 threshold candidates


# 7) Hand-off Artifacts for Team


In [ ]:
# Save outputs used by Member B/C
# e.g., cleaned splits, score files, figures
